In [2]:
import os
import sys
import pathlib
import numpy as np
import pandas as pd
import seaborn as sns
from ast import literal_eval
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)

current = os.getcwd()
parent = os.path.dirname(current)
sys.path.append(parent)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split, TensorDataset, DataLoader

from swp.utils.datasets import get_phoneme_to_id
from swp.utils.models import get_model, load_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vowels = [
    "AH", "OY", "AA", "AY", "ER", "AO", "UW", "IH", 
    "EH", "UH", "IY", "EY", "OW", "AE", "AW"
]

consonants = [
    "TH", "D", "G", "W", "F", "B", "P", "K", "ZH",
    "L", "T", "R", "M", "N", "NG", "Z", "S", "Y", 
    "JH", "SH", "HH", "CH", "V", "DH"
]

converters = {
    "Word": str, 
    "Phonemes": literal_eval, 
    "No_Stress": literal_eval, 
    "Prediction": literal_eval,
    "Real": bool,
}

In [4]:
csv_path = "../results/evaluation/Ua_LSTM_h128_l1_v42_d0.0_t0.0_s1/b1024_l0.001_fall_s42_sn_ec/75/control/trigrams.csv"
data = pd.read_csv(csv_path, index_col=0, converters=converters)

h_path = "../results/evaluation/Ua_LSTM_h128_l1_v42_d0.0_t0.0_s1/b1024_l0.001_fall_s42_sn_ec/75/control/trigrams_h.npy"
emb_h = np.load(h_path)[np.arange(len(data)), data.Length, :]

c_path = "../results/evaluation/Ua_LSTM_h128_l1_v42_d0.0_t0.0_s1/b1024_l0.001_fall_s42_sn_ec/75/control/trigrams_c.npy"
emb_c = np.load(c_path)

In [ ]:
# batch_size = 64
# hidden_size = 128
# p2i = get_phoneme_to_id()

# df = data.copy()
# hs = emb_h.copy()
# cs = emb_c.copy()

# df = df[df["Length"] == 2]
# df["First"] = df["No_Stress"].apply(lambda x: x[0])
# df["Second"] = df["No_Stress"].apply(lambda x: x[1])

# target = "AH" # "B"
# exclude_1 = ["AA", "K"]
# exclude_2 = ["AY", "F"]

# df = df[~df["First"].isin(exclude_1)]
# df1 = df[~df["Second"].isin(exclude_2)]
# X = hs[df1.index]

# df2 = df[df["Second"] == target]
# y = hs[df2.index]
# c = cs[df2.index]
# n = len(df1) / len(df2)
# y = np.repeat(y, int(n), axis=0)
# c = np.repeat(c, int(n), axis=0)

# targets = df2["No_Stress"].apply(lambda x: [p2i[p] for p in x])
# targets = np.repeat(targets.tolist(), int(n), axis=0)

# X = torch.tensor(X, dtype=torch.float32).to(device)
# y = torch.tensor(y, dtype=torch.float32).to(device)
# c = torch.tensor(c, dtype=torch.float32).to(device)
# t = torch.tensor(targets, dtype=torch.long).to(device)

# dataset = TensorDataset(X, y, c, t)
# train_size = int(0.8 * len(dataset))
# valid_size = len(dataset) - train_size
# train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [174]:
batch_size = 64
hidden_size = 128
p2i = get_phoneme_to_id()
stop_token = p2i["<EOS>"]

df = data.copy()
hs = emb_h.copy()
cs = emb_c.copy()

df = df[df["Length"] == 2]
df["P1"] = df["No_Stress"].apply(lambda x: x[0])
df["P2"] = df["No_Stress"].apply(lambda x: x[1])

I = []
Xh = []
Xc = []
Yh = []
Yc = []
T = []

for p in vowels + consonants:
    dfx = df.query(f"P1 == '{p}' & Second != 'V'")
    # print(dfx)
    # break
    dfy = df.query(f"P1 == '{p}' & Second == 'V'")
    n, m = len(dfx), len(dfy)

    i = dfx["No_Stress"].apply(lambda x: [p2i[p] for p in x] + [stop_token]).tolist()

    xh = hs[dfx.index]
    xc = cs[dfx.index]
    # xh = np.repeat(xh, m, axis=0)
    # xc = np.repeat(xc, m, axis=0)
    # xh = np.mean(xh, axis=0)
    # xc = np.mean(xc, axis=0)
    # xh = np.tile(xh, (n * m, 1))
    # xc = np.tile(xc, (n * m, 1))

    yh = hs[dfy.index]
    yc = cs[dfy.index]
    # yh = np.tile(yh, (n, 1))
    # yc = np.tile(yc, (n, 1))
    yh = np.mean(yh, axis=0)
    yc = np.mean(yc, axis=0)
    yh = np.tile(yh, (n, 1))
    yc = np.tile(yc, (n, 1))

    t = dfy["No_Stress"].apply(lambda x: [p2i[p] for p in x] + [stop_token])
    t = np.tile(np.array(t.tolist()), (n, 1))
    t = t[:n]

    I.append(i)
    Xh.append(xh)
    Xc.append(xc)
    Yh.append(yh)
    Yc.append(yc)
    T.append(t)

I = np.vstack(I)
Xh = np.vstack(Xh)
Xc = np.vstack(Xc)
Yh = np.vstack(Yh)
Yc = np.vstack(Yc)
T = np.vstack(T)

I = torch.tensor(I, dtype=torch.long).to(device)
Xh = torch.tensor(Xh, dtype=torch.float32).to(device)
Xc = torch.tensor(Xc, dtype=torch.float32).to(device)
Yh = torch.tensor(Yh, dtype=torch.float32).to(device)
Yc = torch.tensor(Yc, dtype=torch.float32).to(device)
T = torch.tensor(T, dtype=torch.long).to(device)

print(I.shape)
print(Xh.shape)
print(Xc.shape)
print(Yh.shape)
print(Yc.shape)
print(T.shape)

dataset = TensorDataset(I, Xh, Xc, Yh, Yc, T)
train_size = int(0.8 * len(dataset))
valid_size = len(dataset) - train_size
train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

torch.Size([936, 3])
torch.Size([936, 128])
torch.Size([936, 128])
torch.Size([936, 128])
torch.Size([936, 128])
torch.Size([936, 3])


In [175]:
model_name = "Ua_LSTM_h128_l1_v42_d0.0_t0.0_s1"
train_name = "b1024_l0.001_fall_s42_sn_ec"
checkpoint = "75"

model = get_model(model_name)
load_weights(
        model=model,
        model_name=model_name,
        train_name=train_name,
        checkpoint=checkpoint,
        device=device,
    )
decoder = model.decoder.to(device)
decoder.eval()

start_token = torch.Tensor([model.start_token_id])

class CombinedLoss(nn.Module):
    """
    mixed_loss = mse + alpha * (1 - cosine)
    """
    def __init__(self, alpha: float = 0.1, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.mse   = nn.MSELoss(reduction=reduction)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        e_dist = self.mse(pred, target)
        c_dist = 1.0 - F.cosine_similarity(pred, target, dim=1)
        if c_dist.ndim:                     # keep 'mean' behaviour consistent
            c_dist = c_dist.mean()

        # with torch.no_grad():
        #     print("mse", e_dist.item(), "cos", c_dist.item())

        return e_dist + self.alpha * c_dist

In [185]:
l_rate = 1e-3
n_epochs = 150

Cs = [p2i[p] for p in consonants]
Vs = [p2i[p] for p in vowels]
i2p = {v: k for k, v in p2i.items()}

class Net(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        
        with torch.no_grad():
            self.fc1.weight.copy_(torch.eye(hidden_size))   # W ← I
            self.fc1.bias.zero_()                           # b ← 0

        # freeze the weight, leave the bias trainable
        self.fc1.weight.requires_grad_(False)

    def forward(self, x):
        x = self.fc1(x)
        # x = F.relu(x)
        return x
    
model = Net(hidden_size).to(device)
criterion = nn.MSELoss()
# criterion = CombinedLoss(alpha=0.0)

# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
optimizer = torch.optim.AdamW(model.parameters(), lr=l_rate, weight_decay=1e-4)

for epoch in range(1, n_epochs+1):
    model.train()
    running_loss = 0.0

    for _, xh, _, yh, _, _ in train_loader:
        xh = xh.to(device)
        yh = yh.to(device)

        optimizer.zero_grad()
        hidden = model(xh)
        loss = criterion(hidden, yh)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xh.size(0)
    
    epoch_loss = running_loss / len(train_dataset)

    model.eval()
    correct = 0
    total = 0

    s_correct = 0
    e_correct = 0

    counts = {
        "CC" : 0,
        "CV" : 0,
        "VC" : 0,
        "VV" : 0,
    }

    for inputs, xh, _, yh, yc, target in valid_loader:
        inputs = inputs.to(device)
        xh = xh.to(device)
        yh = yh.to(device)
        yc = yc.to(device)
        target = target.to(device)

        start = (
            start_token
            .repeat((xh.size(0), 1))
            .to(device, dtype=torch.int)
        )
        hidden = model(xh).unsqueeze(0)
        cell = yc.unsqueeze(0)

        preds = decoder(start, (hidden, cell), target)
        preds = preds.argmax(dim=-1)
        # print(preds.shape, target.shape)
        # print(preds[:, 2])
        
        correct += np.isin(preds[:, 1], Vs).sum()
        s_correct += (preds[:, 0] == target[:, 0]).sum()
        e_correct += np.isin(preds[:, 2], [stop_token]).sum()
        total += target.size(0)

        # correct += (preds == target).sum().item()
        # total += target.size(0) * target.size(1)

        # get only incorrect predictions for printing
        if epoch == n_epochs:
            mask = ~np.isin(preds[:, 1], Vs)
            inputs = inputs[mask]
            preds = preds[mask]

            wrong_i = [[i2p[p] for p in b] for b in inputs.tolist()]
            wrong_p = [[i2p[p] for p in b] for b in preds.tolist()]

            for i, p in zip(wrong_i, wrong_p):
                print(" ".join(i), "   ->   ", " ".join(p))
                if i[0] in consonants and i[1] in consonants:
                    counts["CC"] += 1
                elif i[0] in consonants and i[1] in vowels:
                    counts["CV"] += 1
                elif i[0] in vowels and i[1] in consonants:
                    counts["VC"] += 1
                elif i[0] in vowels and i[1] in vowels:
                    counts["VV"] += 1
        
    accuracy = correct / total
    s_accuracy = s_correct / total
    e_accuracy = e_correct / total

    if epoch % 10 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch}/{n_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy:.4f}, End: {e_accuracy:.4f}, Sanity: {s_accuracy:.4f}")
    
    if epoch == n_epochs:
        print(counts, total)


Epoch 10/150, Loss: 0.1011, Accuracy: 0.7553, End: 0.7553, Sanity: 1.0000
Epoch 20/150, Loss: 0.0929, Accuracy: 0.7660, End: 0.7660, Sanity: 1.0000
Epoch 30/150, Loss: 0.0898, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 40/150, Loss: 0.0886, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 50/150, Loss: 0.0881, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 60/150, Loss: 0.0879, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 70/150, Loss: 0.0878, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 80/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 90/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 100/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 110/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 120/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 130/150, Loss: 0.0877, Accuracy: 0.7926, End: 0.7926, Sanity: 1.0000
Epoch 140/150, Loss: 0.0877, Accur